# Solubilidad molecular: regresión lineal vs. bosque aleatorio (Random Forest)


**Objetivo:** cargar datos reales, separar características y objetivo, entrenar **dos modelos** (regresión lineal y Random Forest), evaluar con **MSE** y **R²**, y **comparar e interpretar** los resultados.

**Requisitos:** `pandas`, `numpy`, `scikit-learn`, `matplotlib` (opcional para gráficos).

---
## Significado de cada columna del dataset

El archivo **Delaney solubility with descriptors** contiene moléculas y un descriptor de **solubilidad en agua**, junto con propiedades calculadas (descriptores moleculares). En química/medio ambiente y farmacia, la solubilidad influye en absorción, formulación y toxicidad.

| Columna | Nombre en inglés | Qué representa (resumen) |
|---------|-------------------|---------------------------|
| **logS** | log solubility | **Variable objetivo (`y`)**. Solubilidad expresada en escala logarítmica (valores más altos suelen indicar mayor solubilidad en el sentido del dataset). Es un número **continuo** → problema de **regresión**. |
| **MolLogP** | Molecular LogP | **Descriptor lipofílico** (partición octanol/agua en escala log). Indica qué tan “grasa” o hidrófoba tiende a ser la molécula. |
| **MolWt** | Molecular weight | **Peso molecular** (masa aproximada de la molécula). |
| **NumRotatableBonds** | Number of rotatable bonds | **Número de enlaces simples rotativos** (flexibilidad conformacional; moléculas más flexibles suelen tener más). |
| **AromaticProportion** | Aromatic proportion | **Proporción de átomos** en el sistema que son **aromáticos** (p. ej. anillos bencénicos), entre 0 y 1 aproximadamente. |

**Resumen:** Las cuatro primeras columnas (sin contar el índice) son **características de entrada (`X`)**; **logS** es lo que el modelo intenta **predecir**.

---
## Paso 1 — Cargar y explorar los datos

**Documentación:**  
En aprendizaje supervisado siempre conviene **inspeccionar** el tamaño del conjunto, las primeras filas y los tipos de datos antes de modelar.

**Qué debes hacer:**

1. Importar `pandas`.
2. Leer el CSV desde la URL (misma fuente que en las diapositivas).
3. Mostrar `shape`, `head()` y, si quieres, `info()` o `describe()`.

**Línea de referencia (ejemplo antes de tu código):**

```python
df = pd.read_csv('https://raw.githubusercontent.com/dataprofessor/data/master/delaney_solubility_with_descriptors.csv')
```

En las diapositivas aparece `print(df.shape)  # aprox. (1144, 5)` — comprueba que obtienes un número similar de filas y columnas.

In [8]:
import pandas as pd

url = "https://raw.githubusercontent.com/dataprofessor/data/master/delaney_solubility_with_descriptors.csv"

# Completa: carga el DataFrame y explora los datos
df = pd.read_csv(url)
print(df.shape)
df.head()

ModuleNotFoundError: No module named 'pandas'

---
## Paso 2 — Separar características (`X`) y objetivo (`y`)

**Documentación:**

- **`y`**: la variable que quieres predecir → en este dataset, **`logS`**.
- **`X`**: todas las columnas que usas como entradas → las descriptores moleculares; en pandas suele hacerse **eliminando** la columna del objetivo del DataFrame completo.

**Líneas de referencia (ejemplo):**

```python
y = df['logS']
X = df.drop('logS', axis=1)
```

**Completa** la celda siguiente con estas asignaciones (o equivalentes) y comprueba `X.shape`, `y.shape`.

In [ ]:
# y = ...
# X = ...
# print(X.shape, y.shape)

---
## Paso 3 — Partición entrenamiento / prueba

**Documentación:**

- **Entrenamiento (~80%)**: el modelo **aprende** patrones.
- **Prueba (~20%)**: datos **no vistos** durante el entrenamiento; sirve para estimar el rendimiento en datos nuevos y detectar **sobreajuste** (memorizar en lugar de generalizar).

Usa `train_test_split` de `sklearn.model_selection` con `test_size=0.2` y `random_state=100` (como en las diapositivas) para **reproducibilidad**.

**Línea de referencia (ejemplo):**

```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)
```

In [ ]:
from sklearn.model_selection import train_test_split

# Completa la partición
# X_train, X_test, y_train, y_test = train_test_split(...)

---
## Paso 4 — Regresión lineal

**Documentación:**

La **regresión lineal** busca una combinación lineal de las características que minimice el error (método de **mínimos cuadrados** en `sklearn`):

$$\text{logS} \approx \beta_0 + \beta_1\,\text{MolLogP} + \beta_2\,\text{MolWt} + \beta_3\,\text{NumRotatableBonds} + \beta_4\,\text{AromaticProportion}$$

- $\beta_0$: **intercepto** (valor base).
- $\beta_1, \ldots, \beta_4$: **coeficientes** (cuánto cambia la predicción al subir una unidad cada característica, manteniendo el resto fijo en el modelo lineal estándar).

**Ventajas:** rápida, fácil de interpretar, buena **línea base**.  
**Limitaciones:** asume relaciones **lineales**; sensible a **valores atípicos**; no captura interacciones complejas por sí sola.

**Líneas de referencia (ejemplo):**

```python
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_lr_train_pred = lr.predict(X_train)
y_lr_test_pred = lr.predict(X_test)
```

In [ ]:
from sklearn.linear_model import LinearRegression

# Crea el modelo, entrena y predice en train y test
# lr = LinearRegression()
# lr.fit(...)
# y_lr_train_pred = ...
# y_lr_test_pred = ...

---
## Paso 5 — Random Forest (regresión)

**Documentación:**

**Random Forest** combina muchos **árboles de decisión**; cada árbol hace una predicción y el resultado final es el **promedio** de las predicciones (en regresión). Puede capturar **relaciones no lineales** e interacciones entre variables.

En las diapositivas se usa `RandomForestRegressor(max_depth=2, random_state=100)` para **limitar la profundidad** de cada árbol y reducir riesgo de sobreajuste (árboles muy profundos memorizan más).

**Ventajas:** flexible, a menudo robusto, ofrece **importancia de variables**.  
**Limitaciones:** menos interpretable que la recta/hiperplano lineal, más costoso de entrenar, puede sobreajustar si no se regulan hiperparámetros.

**Líneas de referencia (ejemplo):**

```python
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(max_depth=2, random_state=100)
rf.fit(X_train, y_train)
y_rf_train_pred = rf.predict(X_train)
y_rf_test_pred = rf.predict(X_test)
```

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Crea el bosque, entrena y predice en train y test
# rf = RandomForestRegressor(max_depth=2, random_state=100)
# rf.fit(...)
# y_rf_train_pred = ...
# y_rf_test_pred = ...

---
## Paso 6 — Métricas: MSE y R²

**Documentación:**

**MSE (error cuadrático medio):**  
$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$  
- Penaliza más los errores grandes (al elevar al cuadrado).  
- **Menor MSE** en prueba → mejor ajuste en general.  
- Unidades: al cuadrado de la variable objetivo (menos intuitivo que el error absoluto).

**R² (coeficiente de determinación):**  
Mide qué fracción de la **varianza** de `y` explica el modelo respecto a predecir siempre la **media** de `y`.

- **Mayor R²** (hasta 1.0) → mejor en este sentido.  
- R² = 0: tan solo como predecir la media.  
- R² negativo: peor que la media.

**Regla orientativa (solo guía, depende del dominio):** R² > 0.7 suele considerarse razonablemente bueno en muchos contextos introductorios; valores muy cercanos a 1 en datos reales merecen revisar sobreajuste.

**Líneas de referencia (ejemplo):**

```python
from sklearn.metrics import mean_squared_error, r2_score
lr_test_mse = mean_squared_error(y_test, y_lr_test_pred)
lr_test_r2 = r2_score(y_test, y_lr_test_pred)
# Igual para Random Forest con y_rf_test_pred
```

**Completa** el cálculo para **entrenamiento y prueba** de ambos modelos e imprime los resultados.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# Calcula MSE y R² para LR y RF (train y test)
# Imprime tablas o líneas comparativas
# Ejemplo de formato:
# print(f"LR  test  MSE: ...  R²: ...")
# print(f"RF  test  MSE: ...  R²: ...")

---
## Paso 7 — Comparar modelos e interpretar resultados

### Cómo comparar

| Criterio | Qué mirar |
|----------|-----------|
| **Generalización** | Métricas en el conjunto de **prueba** (no solo en entrenamiento). |
| **MSE en prueba** | ¿Cuál es **menor**? Ese modelo comete errores cuadráticos medios más pequeños. |
| **R² en prueba** | ¿Cuál es **mayor**? Ese modelo explica más varianza de `logS` en datos no vistos. |
| **Brecha train vs test** | Si en entrenamiento es muy bueno y en prueba mucho peor → posible **sobreajuste**. Si son parecidos → mejor señal de estabilidad. |

### Qué esperar según las diapositivas (referencia)

En el material original, con `max_depth=2` en Random Forest, a menudo la **regresión lineal** obtiene **mejor R² en prueba** y **menor MSE en prueba** que el bosque en este dataset concreto. Eso **no** significa que la lineal sea siempre superior: aquí la relación entre descriptores y `logS` es bastante **lineal** y los datos están **limpios**, así que un modelo simple encaja bien. Random Forest con poca profundidad puede **subajustar** (no captura suficiente señal).

### Diferencias conceptuales (resumen)

- **Regresión lineal:** una fórmula explícita; coeficientes interpretables directamente; asume linealidad.
- **Random Forest:** muchas reglas en forma de árbol; predicción = promedio de árboles; captura no linealidades pero es más “caja oscura” y depende de hiperparámetros (`max_depth`, `n_estimators`, etc.).

### Cómo explicar R² a alguien no técnico (idea)

“De la variabilidad de la solubilidad en los datos de prueba, el modelo explica aproximadamente un **X %**” (si R² = 0.79 → ~79 % de la varianza explicada, en sentido estadístico; no implica causalidad química directa).

---

**Tarea:** En la celda siguiente, construye una **tabla** (por ejemplo un `DataFrame` de pandas) con columnas: método, MSE train, R² train, MSE test, R² test. Luego escribe en **2–3 frases** cuál modelo elegirías para este problema y por qué, según **tus** números.

In [ ]:
# Opcional: tabla comparativa con pandas
# import pandas as pd
# comparacion = pd.DataFrame({...})
# print(comparacion)

# Escribe aquí tu interpretación en comentarios o en un print de texto:
# print("Mi conclusión: ...")

---
## Paso 8 (opcional) — Gráfico: valores reales vs predichos en prueba

**Documentación:**  
Un gráfico de dispersión de `y_test` frente a predicciones ayuda a ver si los puntos se agrupan cerca de la diagonal (predicción perfecta sería y = ŷ).

**Línea de referencia (idea):**

```python
import matplotlib.pyplot as plt
plt.scatter(y_test, y_lr_test_pred, alpha=0.5, label='Linear Regression')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
```

In [ ]:
# import matplotlib.pyplot as plt
# Gráfico y_test vs y_lr_test_pred y/o y_rf_test_pred
# plt.xlabel('logS real (prueba)')
# plt.ylabel('logS predicho')
# plt.legend()
# plt.show()

---
## Preguntas de reflexión (del material teórico)

1. ¿Por qué crees que la regresión lineal puede superar a Random Forest en **este** dataset con `max_depth=2`?
2. ¿Qué pasaría si aumentas mucho `max_depth` o `n_estimators` en Random Forest?
3. ¿Qué otras propiedades moleculares podrían mejorar las predicciones?
4. ¿Con qué cautela usarías el modelo para **moléculas nuevas** fuera del rango químico del conjunto de entrenamiento?

---

## Lista de comprobación

- [ ] Datos cargados y columnas entendidas
- [ ] `X` e `y` definidos correctamente
- [ ] Partición train/test con `random_state=100`
- [ ] Regresión lineal entrenada y predicciones en train y test
- [ ] Random Forest entrenado con `max_depth=2` y `random_state=100`
- [ ] MSE y R² calculados para ambos modelos (train y test)
- [ ] Comparación escrita: cuál es mejor en prueba y por qué
- [ ] (Opcional) Gráfico real vs predicho

---

**Próximos pasos sugeridos:** probar otros `max_depth` y `n_estimators`, validación cruzada, u otros algoritmos (SVR, Gradient Boosting).